# Exercise 4: Prompting and Extraction

**David Jorgensen**  
Department of Data Science, Bellevue University  
DSC670: Advanced Uses of Generative AI  

---

## Setup & Environment Configuration

This cell resolves `GEMINI_API_KEY` from the system environment, local `.env` files, or directly from `~/.bashrc` as a defensive fallback if Jupyter was started without inheriting the interactive shell's environment.

In [3]:
import os
import sys
import json
from pathlib import Path
from google import genai
from google.genai import types

# 1. Resolve GEMINI_API_KEY with defensive fallback (env var -> local .env -> ~/.bashrc)
api_key = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")

if not api_key:
    for env_candidate in [Path(".env"), Path("../.env"), Path("../../.env")]:
        if env_candidate.exists():
            with open(env_candidate, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if line.startswith("GEMINI_API_KEY=") or line.startswith("export GEMINI_API_KEY="):
                        api_key = line.split("=", 1)[1].strip().strip("'\"")
                        break
        if api_key:
            print(f"[INFO] Loaded GEMINI_API_KEY from {env_candidate.resolve()}")
            break

if not api_key:
    bashrc_path = os.path.expanduser("~/.bashrc")
    if os.path.exists(bashrc_path):
        with open(bashrc_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line.startswith("export GEMINI_API_KEY=") or line.startswith("GEMINI_API_KEY="):
                    api_key = line.split("=", 1)[1].strip().strip("'\"")
                    print("[INFO] Loaded GEMINI_API_KEY from ~/.bashrc.")
                    break

if not api_key:
    raise EnvironmentError(
        "GEMINI_API_KEY not found. Please ensure it is exported in ~/.bashrc or defined in a .env file."
    )

os.environ["GEMINI_API_KEY"] = api_key
masked_key = api_key[:4] + "..." + api_key[-4:] if len(api_key) > 8 else "****"
print(f"[SUCCESS] Active Gemini API Key detected: {masked_key}")

# Initialize official Google GenAI Client
client_gemini = genai.Client(api_key=api_key)
MODEL_ID = "gemini-3.8-flash"
print(f"[CONFIG] Model Selected: {MODEL_ID}")
print(f"Runtime Python Version:  {sys.version.split()[0]}")


[INFO] Loaded GEMINI_API_KEY from ~/.bashrc.
[SUCCESS] Active Gemini API Key detected: AQ.A...ZQHQ
[CONFIG] Model Selected: gemini-3.8-flash
Runtime Python Version:  3.13.15


## Problem 1: Math and Extraction with Zero and One-Shot Prompting

### Zero-Shot Prompting
We are tasked with extracting the order information from the first email into a specific JSON format, calculating prices, subtotals, and the grand total.

In [4]:
first_email = """Hey Jim,

I'd like to order some new shoes. Please ship the following:

1. Nike Air Jordan I - 2 pair
2. Converse All-Star - 10 pair
3. New Balance 990 - 1 pair
4. Nike Zoom Fly 5 men's red - 2 pair

Please provide sub-totals and grand total cost.

Thanks.

Lance Gentry
123 Main St.
Chelsea, MI 48109
248-229-2229"""

zero_shot_prompt = f"""Extract the order information from the email below and return it as a JSON object.
Make sure to assign reasonable prices for the shoes, calculate the subtotals for each item, and the grand total.

Extract the following fields:
- shoe_agent_name (the person being emailed)
- request_type (e.g. 'Order')
- customer_name
- customer_street
- customer_city
- customer_state
- customer_zip
- customer_phone
- items (a list of objects with shoe_brand, shoe_model, shoe_quantity, shoe_price, shoe_subtotal)
- grand_total

Email:
{first_email}
"""

response = client_gemini.models.generate_content(
    model=MODEL_ID,
    contents=zero_shot_prompt,
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
    ),
)
print("Zero-Shot Output:")
print(response.text)


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Zero-Shot Output:
{
  "shoe_agent_name": "Jim",
  "request_type": "Order",
  "customer_name": "Lance Gentry",
  "customer_street": "123 Main St.",
  "customer_city": "Chelsea",
  "customer_state": "MI",
  "customer_zip": "48109",
  "customer_phone": "248-229-2229",
  "items": [
    {
      "shoe_brand": "Nike",
      "shoe_model": "Air Jordan I",
      "shoe_quantity": 2,
      "shoe_price": 180.00,
      "shoe_subtotal": 360.00
    },
    {
      "shoe_brand": "Converse",
      "shoe_model": "All-Star",
      "shoe_quantity": 10,
      "shoe_price": 65.00,
      "shoe_subtotal": 650.00
    },
    {
      "shoe_brand": "New Balance",
      "shoe_model": "990",
      "shoe_quantity": 1,
      "shoe_price": 200.00,
      "shoe_subtotal": 200.00
    },
    {
      "shoe_brand": "Nike",
      "shoe_model": "Zoom Fly 5 men's red",
      "shoe_quantity": 2,
      "shoe_price": 160.00,
      "shoe_subtotal": 320.00
    }
  ],
  "grand_total": 1530.00
}


### One-Shot Prompting
Now, we use the extracted structure of the first email to construct a one-shot prompt, guiding the model on exactly how to structure the extraction and calculate totals for the second email.

In [5]:
second_email = """Hey Michelle,

I'd like to order some new shoes. Please ship the following:

1. Nike Air Jordan I - 1 pair
2. Converse All-Star - 20 pair
3. New Balance 990 - 2 pair
4. Nike Zoom Fly 5 men's red - 5 pair

Please provide sub-totals and grand total cost.

Thanks.

Artis Gilmore
723 Lexington Blvd.
New York, NY 10001
(503) 484-1029"""

one_shot_example = """{
'shoe_agent_name': 'Jim', 
'request_type': 'Order', 
'customer_name': 'Lance Gentry', 
'customer_street': '123 Main St.', 
'customer_city': 'Chelsea', 
'customer_state': 'MI', 
'customer_zip': 48109, 
'customer_phone': '248-229-2229', 

'items': [
{'shoe_brand': 'Nike', 
'shoe_model': 'Air Jordan I', 
'shoe_quantity': 2, 
'shoe_price': 100, 
'shoe_subtotal': 200}, 
{'shoe_brand': 'Converse', 
'shoe_model': 'All-Star', 
'shoe_quantity': 10, 
'shoe_price': 30, 
'shoe_subtotal': 300}, 
{'shoe_brand': 'New Balance', 
'shoe_model': '990', 
'shoe_quantity': 1, 
'shoe_price': 110, 
'shoe_subtotal': 110}, 
{'shoe_brand': 'Nike', 
'shoe_model': "Zoom Fly 5 men's red", 
'shoe_quantity': 2, 
'shoe_price': 400, 
'shoe_subtotal': 800}], 
'grand_total': 1410}
"""

one_shot_prompt = f"""Extract the order information from the email below and return it as a JSON object.
Make sure to calculate prices for the shoes (you can use similar prices from the example below), the subtotals for each item, and the grand total.

Extract the following fields in the exact same format as the example:
- shoe_agent_name (the person being emailed)
- request_type (e.g. 'Order')
- customer_name
- customer_street
- customer_city
- customer_state
- customer_zip
- customer_phone
- items (a list of objects with shoe_brand, shoe_model, shoe_quantity, shoe_price, shoe_subtotal)
- grand_total

Example Input:
{first_email}

Example Output:
{one_shot_example}

Now extract the information for the following email:
{second_email}
"""

response_one_shot = client_gemini.models.generate_content(
    model=MODEL_ID,
    contents=one_shot_prompt,
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
    ),
)
print("One-Shot Output:")
print(response_one_shot.text)


One-Shot Output:
{
  "shoe_agent_name": "Michelle",
  "request_type": "Order",
  "customer_name": "Artis Gilmore",
  "customer_street": "723 Lexington Blvd.",
  "customer_city": "New York",
  "customer_state": "NY",
  "customer_zip": 10001,
  "customer_phone": "(503) 484-1029",
  "items": [
    {
      "shoe_brand": "Nike",
      "shoe_model": "Air Jordan I",
      "shoe_quantity": 1,
      "shoe_price": 100,
      "shoe_subtotal": 100
    },
    {
      "shoe_brand": "Converse",
      "shoe_model": "All-Star",
      "shoe_quantity": 20,
      "shoe_price": 30,
      "shoe_subtotal": 600
    },
    {
      "shoe_brand": "New Balance",
      "shoe_model": "990",
      "shoe_quantity": 2,
      "shoe_price": 110,
      "shoe_subtotal": 220
    },
    {
      "shoe_brand": "Nike",
      "shoe_model": "Zoom Fly 5 men's red",
      "shoe_quantity": 5,
      "shoe_price": 400,
      "shoe_subtotal": 2000
    }
  ],
  "grand_total": 2920
}


### Commentary on Problem 1
**Zero-Shot Prompting Evaluation**: Zero-shot extraction with Gemini performs well, especially when configured with `response_mime_type="application/json"`. The model successfully identifies entity boundaries, infers realistic market prices for the shoes, and computes arithmetic subtotals and grand totals without explicit manual rules. However, without an anchoring example, schema details such as casing (`shoe_quantity` vs `shoeQuantity`), string-vs-integer typing (`"100"` vs `100`), or item hierarchy can fluctuate across runs.

**One-Shot Prompting Evaluation**: Providing an explicit one-shot example eliminates ambiguity. The model anchors both the structural schema (exact keys, nested list format, integer datatypes) and the mathematical relationship (`shoe_quantity * shoe_price = shoe_subtotal`). It also inherits price consistency across calls (e.g., maintaining similar baseline prices for known models) while accurately recalculating subtotals for the new quantities.

## Problem 2: Chat Completion for Chain of Thought
We recreate the Chain-of-Thought (CoT) chat completion from Listing 6.4 using the Gemini API. We structure multi-turn conversational dialogue using `types.Content` objects (`user` and `model` roles) to provide demonstrative step-by-step reasoning.

In [6]:
# Chain of Thought using Gemini Multi-Turn Content
messages = [
    types.Content(role="user", parts=[types.Part.from_text(text="There were nine computers in the server room. Five more computers were installed each day, from Monday to Thursday. How many computers are now in the server room?")]),
    types.Content(role="model", parts=[types.Part.from_text(text="There are 4 days from Monday to Thursday. 5 computers were added each day. That means in total 4 * 5 = 20 computers were added. There were 9 computers initially, so now there are 9 + 20 = 29 computers. The answer is 29.")]),
    
    types.Content(role="user", parts=[types.Part.from_text(text="Michael had 58 golf balls. On Tuesday, he lost 23 golf balls. On Wednesday, he lost 2 more. How many golf balls did he have at the end of Wednesday?")]),
    types.Content(role="model", parts=[types.Part.from_text(text="Michael initially had 58 balls. He lost 23 on Tuesday, so after that he has 58 - 23 = 35 balls. On Wednesday he lost 2 more so now he has 35 - 2 = 33 balls. The answer is 33.")]),
    
    types.Content(role="user", parts=[types.Part.from_text(text="Olivia has $23. She bought five bagels for $3 each. How much money does she have left?")]),
    types.Content(role="model", parts=[types.Part.from_text(text="She bought 5 bagels for $3 each. This means she spent $15. She has $8 left.")]),
    
    types.Content(role="user", parts=[types.Part.from_text(text="When I was 6 my sister was half my age. Now I'm 70 how old is my sister?")])
]

response_chat = client_gemini.models.generate_content(
    model=MODEL_ID,
    contents=messages,
    config=types.GenerateContentConfig(
        temperature=0.0,
    )
)

print("Answer for the final question:")
print(response_chat.text)


Answer for the final question:
When you were 6, your sister was half your age, which means she was 3 years old (3 years younger than you). 

Since the age difference always stays the same, now that you are 70, she is 70 - 3 = 67 years old. 

The answer is 67.


### Commentary on Problem 2 and Comparison

**Chain of Thought (CoT) Evaluation**: 
Chain-of-Thought prompting conditions the language model to decompose multi-step problems into intermediate linguistic reasoning steps prior to generating the final conclusion. The target question (*"When I was 6 my sister was half my age. Now I'm 70 how old is my sister?"*) is a semantic trap. In naive zero-shot generation without CoT, models can trigger an oversimplified heuristic (`70 / 2 = 35`). With the conversational examples showing step-by-step arithmetic deduction, Gemini correctly establishes the fixed temporal delta (when speaker was 6, sister was 3 $\rightarrow$ delta is 3 years) and accurately computes the answer as $70 - 3 = 67$.

---

### Comparative Analysis: Problem 1 vs. Problem 2 Responses

#### 1. Similarities
- **Contextual In-Context Learning**: Both approaches rely on in-context examples provided within the prompt context window rather than model fine-tuning. Problem 1 uses a single-shot demonstration to calibrate extraction, while Problem 2 uses few-shot dialogue turns to prime mathematical reasoning.
- **Embedded Mathematical Computation**: Both tasks require the model to perform arithmetic operations dynamically. In Problem 1, the model computes unit multiplication and additive grand totals; in Problem 2, the model evaluates sequential additions, subtractions, and rate calculations.
- **Pattern Induction**: In both problems, the model accurately identifies and mirrors the semantic patterns demonstrated in the prompt examples (tabular JSON schema matching vs. question-breakdown-answer narrative flow).

#### 2. Differences
- **Output Modality & Schema Rigidity**:
  - **Problem 1**: Yields a strictly constrained, machine-readable JSON object conforming to an exact structural schema.
  - **Problem 2**: Produces freeform, human-readable natural language prose where semantic transparency and narrative explanation are prioritized.
- **Purpose of Demonstration Examples**:
  - **Problem 1 (Structural Constraint)**: The demonstration acts as a syntactic template, enforcing field keys, nesting depth, and data types (e.g., ensuring numeric fields remain unquoted integers).
  - **Problem 2 (Cognitive Constraint)**: The demonstrations act as an algorithmic blueprint, conditioning the model's autoregressive token prediction to emit intermediate rationale ("scratchpad reasoning") before committing to a final numerical answer.
- **Implicit vs. Explicit Computation**:
  - In **Problem 1**, calculation occurs implicitly within the model's internal representation, with only the evaluated results (`shoe_subtotal`, `grand_total`) exposed in the output.
  - In **Problem 2**, the arithmetic logic is explicitly articulated in the visible output text (`6 / 2 = 3`, `70 - 3 = 67`), enabling full audibility and human verification of the reasoning pathway.